## Cell 1: Package Setup

In [1]:
import sys
print(f'Python: {sys.version}')
!{sys.executable} -m pip install -q transformers==5.9.0 peft==0.19.1 accelerate==1.13.0 bitsandbytes==0.49.2 trl datasets timm einops scikit-learn nltk rouge-score sacrebleu Pillow pandas matplotlib seaborn
print('Done. RESTART KERNEL NOW, then start from Cell 2.')

Python: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
Done. RESTART KERNEL NOW, then start from Cell 2.


## Cell 2: Paths & Config

In [1]:
import os, torch

DATA_DIR    = r'C:\Users\faisal\Downloads\Ahasan\Thesis\data\iu_xray'
WORKING_DIR = r'C:\Users\faisal\Downloads\Ahasan\Thesis\BanglaCXR_Ablation'
os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(os.path.join(WORKING_DIR, 'models'), exist_ok=True)

REPORTS_CSV     = os.path.join(DATA_DIR, 'indiana_reports.csv')
PROJECTIONS_CSV = os.path.join(DATA_DIR, 'indiana_projections.csv')
IMAGES_DIR      = os.path.join(DATA_DIR, 'images_normalized')
MODEL_DIR       = os.path.join(WORKING_DIR, 'models')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'REPORTS_CSV exists:     {os.path.exists(REPORTS_CSV)}')
print(f'PROJECTIONS_CSV exists: {os.path.exists(PROJECTIONS_CSV)}')
print(f'IMAGES_DIR exists:      {os.path.exists(IMAGES_DIR)}')

Device: cuda
REPORTS_CSV exists:     True
PROJECTIONS_CSV exists: True
IMAGES_DIR exists:      True


## Cell 3: GPU Check 

In [2]:
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No GPU found — training will be very slow.')

GPU: NVIDIA GeForce RTX 5050 Laptop GPU
VRAM: 8.5 GB


## Cell 4: Imports & Seeds

In [3]:
import random, re
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import (
    AutoModel, AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Imports and seeds set.")

Imports and seeds set.


## Cell 5: Load & Clean Reports

In [4]:
reports     = pd.read_csv(REPORTS_CSV)
projections = pd.read_csv(PROJECTIONS_CSV)

reports = reports.dropna(subset=['findings', 'impression'], how='all')
reports['findings']   = reports['findings'].fillna('')
reports['impression'] = reports['impression'].fillna('')
reports['report']     = ('Findings: ' + reports['findings'] +
                          '\nImpression: ' + reports['impression'])
reports = reports[reports['report'].str.len() > 30]
print(f'Reports loaded: {len(reports)}')

Reports loaded: 3826


## Cell 6: Filter Frontal & Merge

In [5]:
frontal = projections[
    projections['projection'].str.contains('frontal', case=False, na=False)
].drop_duplicates('uid')

df = pd.merge(frontal[['uid','filename']], reports[['uid','report']], on='uid')
df = df.drop_duplicates('uid').reset_index(drop=True)

# Filter to existing images
df = df[df['filename'].apply(
    lambda f: os.path.exists(os.path.join(IMAGES_DIR, f))
)].reset_index(drop=True)

print(f'Merged dataset: {len(df)} samples')
print(df[['filename','report']].head(2))

Merged dataset: 3666 samples
                 filename                                             report
0  1_IM-0001-4001.dcm.png  Findings: The cardiac silhouette and mediastin...
1  2_IM-0652-1001.dcm.png  Findings: Borderline cardiomegaly. Midline ste...


## Cell 7: Train/Val/Test Split

In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=SEED)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Train: 2566 | Val: 550 | Test: 550


## Cell 8: Load RAD-DINO (Frozen)

In [7]:
RAD_DINO_ID = 'microsoft/rad-dino'
print(f'Loading RAD-DINO from {RAD_DINO_ID}...')

rad_dino = AutoModel.from_pretrained(
    RAD_DINO_ID, trust_remote_code=True
).to(DEVICE, dtype=torch.float16)

for p in rad_dino.parameters():
    p.requires_grad = False
rad_dino.eval()

VISUAL_DIM = rad_dino.config.hidden_size
print(f'RAD-DINO loaded. Visual dim: {VISUAL_DIM}')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading RAD-DINO from microsoft/rad-dino...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

RAD-DINO loaded. Visual dim: 768
VRAM: 0.18 GB


## Cell 9: Baseline Projector (Flat MLP)

In [8]:
class FlatMLPProjector(nn.Module):
    def __init__(self, visual_dim=768, llm_dim=3072):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(visual_dim, llm_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(llm_dim, llm_dim),
            nn.LayerNorm(llm_dim)
        )

    def forward(self, visual_feats):
        # Mean-pool all patch tokens → single context vector
        pooled = visual_feats.mean(dim=1, keepdim=True)   # [B, 1, 768]
        return self.net(pooled.float())                    # [B, 1, 3072]

print("FlatMLPProjector (ablation baseline) defined.")
print("Visual tokens after projection: 1 (pooled)")
print("Compare: AnatomyGuidedProjector produces 32 tokens")

FlatMLPProjector (ablation baseline) defined.
Visual tokens after projection: 1 (pooled)
Compare: AnatomyGuidedProjector produces 32 tokens


## Cell 10: Load Phi-3.5 Mini + QLoRA

In [10]:
PHI_MODEL_ID = 'microsoft/Phi-3.5-mini-instruct'
print(f'Loading Phi-3.5...')

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

llm = AutoModelForCausalLM.from_pretrained(
    PHI_MODEL_ID,
    quantization_config=quant_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation='eager'
)
llm.gradient_checkpointing_enable()
llm = prepare_model_for_kbit_training(llm)
llm.config.use_cache = False

LLM_DIM = llm.config.hidden_size
tokenizer = AutoTokenizer.from_pretrained(
    PHI_MODEL_ID, trust_remote_code=True, padding_side='right'
)
tokenizer.pad_token = tokenizer.eos_token

print(f'Phi-3.5 loaded. Hidden dim: {LLM_DIM}')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading Phi-3.5...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Phi-3.5 loaded. Hidden dim: 3072
VRAM: 4.71 GB


## Cell 11: Apply LoRA

In [11]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)
llm = get_peft_model(llm, lora_config)
llm.print_trainable_parameters()
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

trainable params: 50,331,648 || all params: 3,871,411,200 || trainable%: 1.3001
VRAM: 4.93 GB


## Cell 12: BanglaCXR Ablation Model Class

In [12]:
class BanglaCXRAblation(nn.Module):
    def __init__(self, vision_encoder, projector, llm, tokenizer):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.projector      = projector
        self.llm            = llm
        self.tokenizer      = tokenizer

        PROMPT = (
            '<|user|>\n<|image_1|>\n'
            'You are a radiologist. Generate a structured report.\n'
            'Findings:\nImpression:\n<|end|>\n<|assistant|>\n'
        )
        self.prompt_ids = tokenizer(
            PROMPT, return_tensors='pt', add_special_tokens=False
        ).input_ids

    def forward(self, pixel_values, input_ids, attention_mask, labels):
        vis_feats  = self.vision_encoder(pixel_values).last_hidden_state.float()
        proj_feats = self.projector(vis_feats)             # [B, 1, llm_dim]

        text_embeds = self.llm.get_input_embeddings()(input_ids)
        combined_embeds = torch.cat([proj_feats, text_embeds], dim=1)

        pad_len = proj_feats.shape[1]
        combined_mask = torch.cat([
            torch.ones(attention_mask.shape[0], pad_len,
                       dtype=attention_mask.dtype, device=attention_mask.device),
            attention_mask
        ], dim=1)

        full_labels = torch.full(
            (labels.shape[0], pad_len + labels.shape[1]),
            -100, dtype=labels.dtype, device=labels.device
        )
        full_labels[:, pad_len:] = labels

        return self.llm(
            inputs_embeds=combined_embeds,
            attention_mask=combined_mask,
            labels=full_labels
        )

print("BanglaCXRAblation model class defined.")

BanglaCXRAblation model class defined.


## Cell 13: Dataset

In [13]:
IMG_SIZE   = 518
BATCH_SIZE = 2
MAX_LENGTH = 256

PROMPT_TEXT = (
    '<|user|>\n<|image_1|>\n'
    'You are a radiologist. Generate a structured report.\n'
    'Findings:\nImpression:\n<|end|>\n<|assistant|>\n'
)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class CXRDataset(Dataset):
    def __init__(self, df, images_dir, tokenizer, transform,
                 prompt=PROMPT_TEXT, max_length=MAX_LENGTH):
        self.df         = df
        self.images_dir = images_dir
        self.tokenizer  = tokenizer
        self.transform  = transform
        self.prompt     = prompt
        self.max_length = max_length

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(
            os.path.join(self.images_dir, row['filename'])
        ).convert('RGB')
        pv  = self.transform(img)

        text = self.prompt + row['report']
        enc  = self.tokenizer(
            text, max_length=self.max_length,
            truncation=True, padding='max_length', return_tensors='pt'
        )
        ids  = enc['input_ids'].squeeze(0)
        mask = enc['attention_mask'].squeeze(0)

        prompt_len = len(self.tokenizer(
            self.prompt, add_special_tokens=False
        ).input_ids)
        labels = ids.clone()
        labels[:prompt_len] = -100
        labels[mask == 0]   = -100

        return {'pixel_values': pv, 'input_ids': ids,
                'attention_mask': mask, 'labels': labels}

train_ds = CXRDataset(train_df, IMAGES_DIR, tokenizer, train_transform)
val_ds   = CXRDataset(val_df,   IMAGES_DIR, tokenizer, val_transform)
test_ds  = CXRDataset(test_df,  IMAGES_DIR, tokenizer, val_transform)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')


Train: 2566 | Val: 550 | Test: 550


## Cell 14: Instantiate Model + Weighted Sampler

In [14]:
import re
from torch.utils.data import WeightedRandomSampler

ABNORMAL_PATTERNS = [
    r'(?<!no\s)(?<!without\s)(?<!negative\s)(?<!no evidence of\s)'
    r'(?<!no focal\s)(?<!clear without\s)'
    r'\b(consolidation|effusion|pneumothorax|cardiomegaly|'
    r'atelectasis|infiltrate|opacity|edema|mass|nodule|'
    r'fracture|enlarged heart|thickening|scarring|hyperexpand|'
    r'degenerative|scoliosis|kyphosis|granuloma|calcif)\b'
]
def is_truly_abnormal(report):
    for pattern in ABNORMAL_PATTERNS:
        if re.search(pattern, report.lower()):
            return True
    return False

# Weighted sampler — abnormal 3x (identical to Stage 1)
train_weights = [
    3.0 if is_truly_abnormal(row['report']) else 1.0
    for _, row in train_df.iterrows()
]
sampler = WeightedRandomSampler(
    train_weights, num_samples=len(train_weights), replacement=True
)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0
)
val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=1,
                         shuffle=False, num_workers=0)

# Instantiate ablation model
projector_baseline = FlatMLPProjector(VISUAL_DIM, LLM_DIM).to(DEVICE)
model = BanglaCXRAblation(rad_dino, projector_baseline, llm, tokenizer).to(DEVICE)
model.llm.config.use_cache = False

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total: {total:,} | Trainable: {trainable:,} ({100*trainable/total:.2f}%)')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Optimizer (identical LRs to Stage 1)
NUM_EPOCHS   = 40
LR_PROJECTOR = 3e-4
LR_LORA      = 1e-5
GRAD_ACCUM   = 8

optimizer = torch.optim.AdamW([
    {'params': model.projector.parameters(),
     'lr': LR_PROJECTOR, 'weight_decay': 0.01},
    {'params': [p for p in model.llm.parameters() if p.requires_grad],
     'lr': LR_LORA, 'weight_decay': 0.01}
])
total_steps  = NUM_EPOCHS * len(train_loader) // GRAD_ACCUM
warmup_steps = int(total_steps * 0.05)
scheduler    = get_cosine_schedule_with_warmup(
    optimizer, warmup_steps, total_steps
)
print(f'Steps: {total_steps} | Warmup: {warmup_steps} | '
      f'Eff batch: {BATCH_SIZE * GRAD_ACCUM} | Epochs: {NUM_EPOCHS}')

Total: 2,157,861,120 | Trainable: 62,140,416 (2.88%)
VRAM: 4.98 GB
Steps: 6415 | Warmup: 320 | Eff batch: 16 | Epochs: 40


## Cell 15: Training Loop

In [19]:
#### history  = {'train_loss': [], 'val_loss': []}
best_val = float('inf')

print("Starting ablation (FlatMLP baseline) training...")

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(train_loader,
                desc=f'Epoch {epoch+1:02d}/{NUM_EPOCHS} [Train]',
                leave=False, colour='cyan')

    for step, batch in enumerate(pbar):
        pv   = batch['pixel_values'].to(DEVICE, dtype=torch.float16)
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbl  = batch['labels'].to(DEVICE)

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            out  = model(pv, ids, mask, lbl)
            loss = out.loss / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        train_loss += loss.item()
        pbar.set_postfix({
            'loss': f'{loss.item()*GRAD_ACCUM:.4f}',
            'vram': f'{torch.cuda.memory_allocated()/1e9:.1f}GB'
        })

    avg_train = (train_loss / len(train_loader)) * GRAD_ACCUM
    history['train_loss'].append(avg_train)
    pbar.close()

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            pv   = batch['pixel_values'].to(DEVICE, dtype=torch.float16)
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbl  = batch['labels'].to(DEVICE)
            out  = model(pv, ids, mask, lbl)
            val_loss += out.loss.item()

    avg_val = val_loss / len(val_loader)
    history['val_loss'].append(avg_val)
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | '
          f'Train: {avg_train:.4f} | Val: {avg_val:.4f}')

    if avg_val < best_val:
        best_val = avg_val
        torch.save(model.state_dict(),
                   os.path.join(MODEL_DIR, 'best_model_baseline.pt'))
        print(f'  ✓ Best saved (val={best_val:.4f})')

    # Early stopping — patience 8
    if epoch >= 8:
        recent = history['val_loss'][-8:]
        if min(recent) == recent[0]:
            print(f'\nEarly stop at epoch {epoch+1}.')
            break

print(f'\nTraining complete. Best val: {best_val:.4f}')
print(f'Saved: {os.path.join(MODEL_DIR, "best_model_baseline.pt")}')

Starting ablation (FlatMLP baseline) training...


                                                                                                                       am=4.2GB]

Epoch 01/40 | Train: 0.8962 | Val: 1.5744
  ✓ Best saved (val=1.5744)


                                                                                                                       am=4.2GB]

Epoch 02/40 | Train: 0.8335 | Val: 1.5789


...

Epoch 08/40 | Train: 0.7967 | Val: 1.5793


                                                                                                                       am=4.2GB]

Epoch 09/40 | Train: 0.8012 | Val: 1.5795


                                                                                                                       am=4.2GB]

Epoch 10/40 | Train: 0.7921 | Val: 1.5797

Early stop at epoch 10.

Training complete. Best val: 1.5710
Saved: C:\Users\faisal\Downloads\Ahasan\Thesis\BanglaCXR_Ablation\models\best_model_baseline.pt


## Cell 16: Evaluation

In [24]:
import types, torch, json, os, numpy as np
from PIL import Image
from torchvision import transforms
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import nltk

nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

# Load baseline model
print("Loading baseline checkpoint...")
state = torch.load(os.path.join(MODEL_DIR, 'best_model_baseline.pt'), 
                   map_location='cpu', weights_only=False)
model.load_state_dict(state, strict=False)
del state
torch.cuda.empty_cache()
model.eval()

# Generation function for baseline
def generate_report_baseline(self, pixel_values, max_new_tokens=200):
    self.eval()
    with torch.no_grad():
        vis_feats = self.vision_encoder(pixel_values).last_hidden_state.float()
        proj_feats = self.projector(vis_feats)
        
        prompt = ("<|user|>\<|image_1|>You are a radiologist. Generate a structured report."
                  "Findings:\Impression:\<|end|>\<|assistant|>")
        prompt_ids = self.tokenizer(prompt, return_tensors='pt', 
                                    add_special_tokens=False).input_ids.to(DEVICE)
        prompt_embeds = self.llm.get_input_embeddings()(prompt_ids)
        combined_embeds = torch.cat([proj_feats, prompt_embeds], dim=1)
        
        generated_ids = prompt_ids.clone()
        cur_embeds = combined_embeds
        
        for _ in range(max_new_tokens):
            out = self.llm(inputs_embeds=cur_embeds, use_cache=False)
            next_logits = out.logits[:, -1, :]
            for prev_id in generated_ids[0]:
                next_logits[:, prev_id] /= 1.3
            next_id = next_logits.argmax(dim=-1).unsqueeze(1)
            generated_ids = torch.cat([generated_ids, next_id], dim=1)
            if next_id.item() == self.tokenizer.eos_token_id:
                break
            next_emb = self.llm.get_input_embeddings()(next_id)
            cur_embeds = next_emb if next_emb.dim() == 3 else next_emb.unsqueeze(1)
        
        out_ids = generated_ids[:, prompt_ids.shape[1]:]
        return self.tokenizer.decode(out_ids[0], skip_special_tokens=True)

model.generate_report = types.MethodType(generate_report_baseline, model)

# Generate reports
print(f"\nGenerating {len(test_df)} baseline reports...")
generated_baseline = []
references_baseline = []

tf = transforms.Compose([
    transforms.Resize((518, 518)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

model.eval()
for i, (_, row) in enumerate(test_df.iterrows()):
    img = Image.open(os.path.join(IMAGES_DIR, row['filename'])).convert('RGB')
    pv = tf(img).unsqueeze(0).to(DEVICE, dtype=torch.float16)
    gen = model.generate_report(pv)
    generated_baseline.append(gen.strip())
    references_baseline.append(row['report'])
    if (i+1) % 100 == 0:
        print(f"  [{i+1}/{len(test_df)}]")

# Save baseline outputs
baseline_json = os.path.join(LOG_DIR, 'baseline_english_reports.json')
with open(baseline_json, 'w', encoding='utf-8') as f:
    json.dump({
        'generated': generated_baseline,
        'references': references_baseline
    }, f, indent=2, ensure_ascii=False)
print(f"\n✓ Baseline JSON saved: {baseline_json}")

# Calculate metrics
print("\n" + "="*55)
print("  BASELINE EVALUATION — Flat MLP Projector")
print("="*55)

smoother = SmoothingFunction().method1
refs_tok = [[r.split()] for r in references_baseline]
gens_tok = [g.split() for g in generated_baseline]

bleu1 = corpus_bleu(refs_tok, gens_tok, weights=(1,0,0,0), smoothing_function=smoother)
bleu2 = corpus_bleu(refs_tok, gens_tok, weights=(0.5,0.5,0,0), smoothing_function=smoother)
bleu4 = corpus_bleu(refs_tok, gens_tok, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoother)

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
r1_list, r2_list, rL_list = [], [], []
for gen, ref in zip(generated_baseline, references_baseline):
    s = scorer.score(ref, gen)
    r1_list.append(s['rouge1'].fmeasure)
    r2_list.append(s['rouge2'].fmeasure)
    rL_list.append(s['rougeL'].fmeasure)

meteor = np.mean([meteor_score([r.split()], g.split()) 
                  for r, g in zip(references_baseline, generated_baseline)])

print(f"Test samples  : {len(generated_baseline)}")
print(f"BLEU-1        : {bleu1:.4f}")
print(f"BLEU-2        : {bleu2:.4f}")
print(f"BLEU-4        : {bleu4:.4f}")
print(f"ROUGE-1       : {np.mean(r1_list):.4f}")
print(f"ROUGE-2       : {np.mean(r2_list):.4f}")
print(f"ROUGE-L       : {np.mean(rL_list):.4f}")
print(f"METEOR        : {meteor:.4f}")
print("="*55)

# Show 3 samples
print("\n--- Sample Outputs ---")
for i in range(min(3, len(generated_baseline))):
    print(f"\n[Sample {i+1}]")
    print(f"REF: {references_baseline[i][:250]}...")
    print(f"GEN: {generated_baseline[i][:250]}...")

Loading baseline checkpoint...

Generating 550 baseline reports...
  [100/550]
  [200/550]
  [300/550]
  [400/550]
  [500/550]


NameError: name 'LOG_DIR' is not defined

In [27]:
import json, os
import numpy as np
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import re

# Save generated reports
baseline_json = os.path.join(MODEL_DIR, 'baseline_english_reports.json')
with open(baseline_json, 'w', encoding='utf-8') as f:
    json.dump({
        'generated':  generated_baseline,
        'references': references_baseline
    }, f, indent=2, ensure_ascii=False)
print(f'✓ Saved: {baseline_json}')

# BLEU
smoother = SmoothingFunction().method1
refs_tok  = [[r.split()] for r in references_baseline]
gens_tok  = [g.split()   for g in generated_baseline]

bleu1 = corpus_bleu(refs_tok, gens_tok, weights=(1,0,0,0),             smoothing_function=smoother)
bleu2 = corpus_bleu(refs_tok, gens_tok, weights=(0.5,0.5,0,0),         smoothing_function=smoother)
bleu4 = corpus_bleu(refs_tok, gens_tok, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoother)

# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
r1, r2, rL = [], [], []
for gen, ref in zip(generated_baseline, references_baseline):
    s = scorer.score(ref, gen)
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rL.append(s['rougeL'].fmeasure)

# METEOR
meteor = np.mean([
    meteor_score([r.split()], g.split())
    for r, g in zip(references_baseline, generated_baseline)
])

# Abnormality detection
ABNORMAL_PATTERNS = [
    r'(?<!no\s)(?<!without\s)(?<!negative\s)(?<!no evidence of\s)'
    r'(?<!no focal\s)(?<!clear without\s)'
    r'\b(consolidation|effusion|pneumothorax|cardiomegaly|'
    r'atelectasis|infiltrate|opacity|edema|mass|nodule|'
    r'fracture|enlarged heart|thickening|scarring|hyperexpand|'
    r'degenerative|scoliosis|kyphosis|granuloma|calcif)\b'
]
def is_truly_abnormal(report):
    for p in ABNORMAL_PATTERNS:
        if re.search(p, report.lower()):
            return True
    return False

true_abn = [is_truly_abnormal(r) for r in references_baseline]
pred_abn = [is_truly_abnormal(g) for g in generated_baseline]
tp = sum(t and p for t, p in zip(true_abn, pred_abn))
fp = sum(not t and p for t, p in zip(true_abn, pred_abn))
fn = sum(t and not p for t, p in zip(true_abn, pred_abn))
precision = tp/(tp+fp) if (tp+fp) > 0 else 0
recall    = tp/(tp+fn) if (tp+fn) > 0 else 0
f1        = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0

# Print results
print('\n' + '='*65)
print('   ABLATION COMPARISON TABLE')
print('='*65)
print(f'  {"Metric":<22} {"FlatMLP (baseline)":>20} {"AnatomyGuided (ours)":>20}')
print(f'  {"-"*22} {"-"*20} {"-"*20}')
print(f'  {"BLEU-1":<22} {bleu1:>20.4f} {"0.2884":>20}')
print(f'  {"BLEU-2":<22} {bleu2:>20.4f} {"0.1664":>20}')
print(f'  {"BLEU-4":<22} {bleu4:>20.4f} {"0.0699":>20}')
print(f'  {"ROUGE-1":<22} {np.mean(r1):>20.4f} {"0.4352":>20}')
print(f'  {"ROUGE-2":<22} {np.mean(r2):>20.4f} {"0.1604":>20}')
print(f'  {"ROUGE-L":<22} {np.mean(rL):>20.4f} {"0.3103":>20}')
print(f'  {"Abnormality F1":<22} {f1:>20.4f} {"0.8802":>20}')
print(f'  {"Recall":<22} {recall:>20.4f} {"0.9795":>20}')
print(f'  {"Precision":<22} {precision:>20.4f} {"0.7993":>20}')
print('='*65)

# Sample outputs
print('\n--- Sample Outputs ---')
for i in range(3):
    print(f'\n[Sample {i+1}]')
    print(f'REF: {references_baseline[i][:200]}')
    print(f'GEN: {generated_baseline[i][:200]}')

✓ Saved: C:\Users\faisal\Downloads\Ahasan\Thesis\BanglaCXR_Ablation\models\baseline_english_reports.json

   ABLATION COMPARISON TABLE
  Metric                   FlatMLP (baseline) AnatomyGuided (ours)
  ---------------------- -------------------- --------------------
  BLEU-1                               0.0003               0.2884
  BLEU-2                               0.0000               0.1664
  BLEU-4                               0.0000               0.0699
  ROUGE-1                              0.0147               0.4352
  ROUGE-2                              0.0000               0.1604
  ROUGE-L                              0.0146               0.3103
  Abnormality F1                       0.0000               0.8802
  Recall                               0.0000               0.9795
  Precision                            0.0000               0.7993

--- Sample Outputs ---

[Sample 1]
REF: Findings: The XXXX examination consists of frontal and lateral radiographs of the chest

In [28]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import os

# ── Set Bengali font if available, fallback to default ─────────
import matplotlib.font_manager as fm
bengali_fonts = [f.name for f in fm.fontManager.ttflist
                 if any(x in f.name.lower()
                        for x in ['kalpurush','solaiman','nirmala','vrinda','siyam'])]
if bengali_fonts:
    plt.rcParams['font.family'] = bengali_fonts[0]
    print(f'Bengali font found: {bengali_fonts[0]}')
else:
    print('No Bengali font found — Bengali text will show as boxes in fig3.')
    print('Install SolaimanLipi or Kalpurush for proper rendering.')

SAVE_DIR = os.path.join(WORKING_DIR, 'figures')
os.makedirs(SAVE_DIR, exist_ok=True)

# ── REAL DATA ──────────────────────────────────────────────────
stage1_train = [4.5568,2.1734,1.2864,0.9678,0.7787,0.6992,0.6388,
                0.6354,0.5781,0.5466,0.5341,0.4905,0.4613,0.4365,
                0.4292,0.4202,0.3827,0.3664,0.3577,0.3187,0.3228,
                0.2937,0.2721,0.2577,0.2444]
stage1_val   = [2.6083,1.6051,1.0503,0.8657,0.7698,0.7133,0.6726,
                0.6357,0.6151,0.6060,0.5847,0.5761,0.5652,0.5615,
                0.5635,0.5576,0.5574,0.5516,0.5529,0.5549,0.5562,
                0.5543,0.5585,0.5662,0.5711]

baseline_train = [0.8962,0.8335,0.8227,0.8406,0.7932,
                  0.8184,0.7912,0.7967,0.8012,0.7921]
baseline_val   = [1.5744,1.5789,1.5710,1.5818,1.5773,
                  1.5781,1.5792,1.5793,1.5795,1.5797]

metrics    = ['BLEU-1','BLEU-2','BLEU-4','ROUGE-1','ROUGE-2','ROUGE-L','Abn.\nF1','Recall']
baseline_v = [0.0003, 0.0000, 0.0000, 0.0147,  0.0000,  0.0146,  0.0000, 0.0000]
anatomy_v  = [0.2884, 0.1664, 0.0699, 0.4352,  0.1604,  0.3103,  0.8802, 0.9795]

clinical_labels = ['Cardiomegaly','Pleural\nEffusion','Pneumo-\nthorax',
                   'Consolid-\nation','Atelectasis','Edema',
                   'Lung\nOpacity','Nodule/\nMass','Fracture',
                   'Hyper-\ninflation','Degener-\native','No\nFinding']
clinical_f1 = [0.3158,0.7164,0.7532,0.2895,0.2500,0.1263,
               0.2762,0.0909,0.0000,0.2308,0.1404,0.7962]


# ══════════════════════════════════════════════════════════════
# FIGURE 1: Training Curves
# ══════════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor('white')

e1 = range(1, len(stage1_train)+1)
ax1.plot(e1, stage1_train, 'b-o', ms=5, lw=2, label='Train Loss')
ax1.plot(e1, stage1_val,   'r-s', ms=5, lw=2, label='Val Loss')
ax1.axvline(18, color='#2e7d32', ls='--', lw=2,
            label='Best (Ep.18, val=0.5516)')
ax1.axvline(25, color='#888', ls=':', lw=1.5,
            label='Early stop (Ep.25)')
ax1.fill_between(e1, stage1_train, stage1_val, alpha=0.07, color='purple')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax1.set_title('AnatomyGuided Projector (Ours)\n32 Cross-Attention Region Queries',
              fontsize=11, fontweight='bold', color='#1565c0')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax1.set_xlim(1,25); ax1.set_ylim(0,5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

e2 = range(1, len(baseline_train)+1)
ax2.plot(e2, baseline_train, 'b-o', ms=5, lw=2, label='Train Loss')
ax2.plot(e2, baseline_val,   'r-s', ms=5, lw=2, label='Val Loss')
ax2.axvline(3,  color='#2e7d32', ls='--', lw=2,
            label='Best (Ep.3, val=1.5710)')
ax2.axvline(10, color='#888', ls=':', lw=1.5,
            label='Early stop (Ep.10)')
ax2.fill_between(e2, baseline_train, baseline_val, alpha=0.07, color='red')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax2.set_title('FlatMLP Projector (Ablation Baseline)\nMean-Pooled Single Token',
              fontsize=11, fontweight='bold', color='#b71c1c')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
ax2.set_xlim(1,10); ax2.set_ylim(0,2)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle('BanglaCXR Stage 1 — Training Curves Comparison',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
p1 = os.path.join(SAVE_DIR, 'fig1_training_curves.png')
plt.savefig(p1, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f'✓ {p1}')


# ══════════════════════════════════════════════════════════════
# FIGURE 2: Ablation Bar Chart
# ══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 6.5))
fig.patch.set_facecolor('white')

x     = np.arange(len(metrics))
width = 0.35
bars1 = ax.bar(x-width/2, baseline_v, width,
               label='FlatMLP Baseline (1 pooled token)',
               color='#ef9a9a', edgecolor='#c62828', linewidth=1.2)
bars2 = ax.bar(x+width/2, anatomy_v, width,
               label='AnatomyGuided Projector — Ours (32 region tokens)',
               color='#42a5f5', edgecolor='#0d47a1', linewidth=1.2)

for bar in bars1:
    h = bar.get_height()
    if h > 0.001:
        ax.text(bar.get_x()+bar.get_width()/2., h+0.01,
                f'{h:.4f}', ha='center', va='bottom',
                fontsize=7.5, color='#c62828')
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2., h+0.01,
            f'{h:.4f}', ha='center', va='bottom',
            fontsize=8, fontweight='bold', color='#0d47a1')

ax.annotate('FlatMLP fails completely\n(gibberish output — BLEU≈0)',
            xy=(0-width/2, 0.003), xytext=(1.2, 0.20),
            fontsize=9, color='#c62828',
            arrowprops=dict(arrowstyle='->', color='#c62828', lw=1.5))

ax.set_xlabel('Evaluation Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Ablation Study: AnatomyGuided Projector vs FlatMLP Baseline\n'
             'IU-Xray Test Set (N=550) — All other components identical',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=10)
ax.legend(fontsize=11, loc='upper left')
ax.set_ylim(0, 1.15)
ax.grid(True, axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
p2 = os.path.join(SAVE_DIR, 'fig2_ablation_bars.png')
plt.savefig(p2, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f'✓ {p2}')


# ══════════════════════════════════════════════════════════════
# FIGURE 3: Architecture Diagram
# ══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(18, 6))
ax.set_xlim(0,18); ax.set_ylim(0,6)
ax.axis('off')
fig.patch.set_facecolor('white')

def draw_box(ax, x, y, w, h, lines, fc, ec, fontsize=8.5, tc='white'):
    r = mpatches.FancyBboxPatch((x,y), w, h,
        boxstyle='round,pad=0.15', facecolor=fc,
        edgecolor=ec, linewidth=2, zorder=3)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, '\n'.join(lines),
            ha='center', va='center', fontsize=fontsize,
            fontweight='bold', color=tc, multialignment='center', zorder=4)

def draw_arr(ax, x1, x2, y=3.0):
    ax.annotate('', xy=(x2,y), xytext=(x1,y),
        arrowprops=dict(arrowstyle='->', color='#333333',
                        lw=2.2, mutation_scale=20), zorder=5)

# Background shading
s1bg = mpatches.FancyBboxPatch((0.1,0.8), 12.6, 4.5,
    boxstyle='round,pad=0.2', facecolor='#e3f2fd',
    edgecolor='#1565c0', linewidth=1.5, alpha=0.4, zorder=1)
s2bg = mpatches.FancyBboxPatch((13.1,0.8), 4.7, 4.5,
    boxstyle='round,pad=0.2', facecolor='#fce4ec',
    edgecolor='#880e4f', linewidth=1.5, alpha=0.4, zorder=1)
ax.add_patch(s1bg); ax.add_patch(s2bg)

draw_box(ax, 0.3,  1.8, 1.8, 2.4,
         ['Chest','X-ray','(Input)'], '#546e7a','#263238')
draw_box(ax, 2.5,  1.8, 2.3, 2.4,
         ['RAD-DINO','(Frozen)','ViT-B/16','768-dim'], '#1565c0','#0d47a1')
draw_box(ax, 5.2,  1.8, 2.5, 2.4,
         ['Anatomy-Guided','Projector','32 Cross-Attn','Queries (Novel)'],
         '#6a1b9a','#4a148c')
draw_box(ax, 8.1,  1.8, 2.3, 2.4,
         ['Phi-3.5 Mini','QLoRA r=32','3.8B params','1.3% trainable'],
         '#1b5e20','#003300')
draw_box(ax, 10.8, 1.8, 1.8, 2.4,
         ['English','Radiology','Report'], '#e65100','#bf360c')
draw_box(ax, 13.3, 1.8, 2.3, 2.4,
         ['NLLB-200','Distilled','600M','+Glossary'], '#880e4f','#560027')
draw_box(ax, 13.3, 0.1, 2.3, 1.4,
         ['Bangla Report','(Output)'], '#b71c1c','#7f0000')

draw_arr(ax, 2.1,  2.5)
draw_arr(ax, 4.8,  5.2)
draw_arr(ax, 7.7,  8.1)
draw_arr(ax, 10.4, 10.8)
draw_arr(ax, 12.6, 13.3)
ax.annotate('', xy=(14.45,1.5), xytext=(14.45,1.8),
    arrowprops=dict(arrowstyle='->', color='#333333',
                    lw=2.2, mutation_scale=18), zorder=5)

ax.text(6.5,  5.55, 'Stage 1: Visual-Language Report Generation',
        ha='center', fontsize=12, color='#1565c0', fontweight='bold')
ax.text(14.5, 5.55, 'Stage 2: Translation',
        ha='center', fontsize=11, color='#880e4f', fontweight='bold')
ax.text(6.5,  5.10, '★ Novel component — cross-attention projector proven by ablation',
        ha='center', fontsize=9.5, color='#6a1b9a', style='italic')

ax.set_title(
    'BanglaCXR: Two-Stage Pipeline for Bangla Chest X-ray Report Generation',
    fontsize=14, fontweight='bold', pad=18)

plt.tight_layout()
p3 = os.path.join(SAVE_DIR, 'fig3_architecture.png')
plt.savefig(p3, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f'✓ {p3}')


# ══════════════════════════════════════════════════════════════
# FIGURE 4: Radar Chart
# ══════════════════════════════════════════════════════════════
cats   = ['BLEU-1','BLEU-2','ROUGE-1','ROUGE-L','Abn. F1','Recall','Precision']
base_r = [0.0003, 0.0000, 0.0147, 0.0146, 0.0000, 0.0000, 0.0000]
anat_r = [0.2884, 0.1664, 0.4352, 0.3103, 0.8802, 0.9795, 0.7993]

N      = len(cats)
angles = [n/float(N)*2*np.pi for n in range(N)] + [0]
base_r = base_r + [base_r[0]]
anat_r = anat_r + [anat_r[0]]

fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('white')
ax.set_facecolor('#fafafa')

ax.plot(angles, base_r, 'o-', lw=2.5, color='#e53935',
        label='FlatMLP Baseline')
ax.fill(angles, base_r, alpha=0.12, color='#e53935')
ax.plot(angles, anat_r, 'o-', lw=2.5, color='#1565c0',
        label='AnatomyGuided (Ours)')
ax.fill(angles, anat_r, alpha=0.15, color='#1565c0')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(cats, size=11, fontweight='bold')
ax.set_ylim(0,1)
ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], size=8, color='grey')
ax.grid(color='grey', ls='--', lw=0.5, alpha=0.6)
ax.set_title('Ablation Radar Chart\nAnatomyGuided vs FlatMLP Baseline',
             size=13, fontweight='bold', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.42,1.18), fontsize=11)

plt.tight_layout()
p4 = os.path.join(SAVE_DIR, 'fig4_radar.png')
plt.savefig(p4, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f'✓ {p4}')


# ══════════════════════════════════════════════════════════════
# FIGURE 5: Clinical F1 per Finding
# ══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 5.5))
fig.patch.set_facecolor('white')

colors = ['#ef5350' if f < 0.3 else '#ffa726' if f < 0.5 else '#66bb6a'
          for f in clinical_f1]
bars = ax.bar(clinical_labels, clinical_f1, color=colors,
              edgecolor='#333', linewidth=0.8)
for bar, val in zip(bars, clinical_f1):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.012,
            f'{val:.3f}', ha='center', va='bottom',
            fontsize=8.5, fontweight='bold')

ax.axhline(0.3321, color='#1565c0', ls='--', lw=2,
           label='Macro-F1 = 0.3321')
ax.axhline(0.5897, color='#880e4f', ls='--', lw=2,
           label='Micro-F1 = 0.5897')

red_p    = mpatches.Patch(color='#ef5350', label='F1 < 0.30 (poor)')
orange_p = mpatches.Patch(color='#ffa726', label='0.30 ≤ F1 < 0.50 (moderate)')
green_p  = mpatches.Patch(color='#66bb6a', label='F1 ≥ 0.50 (good)')
line1    = plt.Line2D([0],[0], color='#1565c0', ls='--', lw=2,
                      label='Macro-F1 = 0.3321')
line2    = plt.Line2D([0],[0], color='#880e4f', ls='--', lw=2,
                      label='Micro-F1 = 0.5897')
ax.legend(handles=[red_p, orange_p, green_p, line1, line2],
          fontsize=9, loc='upper right')

ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('BanglaCXR Stage 1 — Clinical Entity F1 per Finding\n'
             '(AnatomyGuided Projector, IU-Xray Test Set N=550)',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.grid(True, axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', labelsize=8.5)

plt.tight_layout()
p5 = os.path.join(SAVE_DIR, 'fig5_clinical_f1.png')
plt.savefig(p5, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f'✓ {p5}')


# ══════════════════════════════════════════════════════════════
# FIGURE 6: Dashboard (all in one — for poster/presentation)
# ══════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Top-left: Stage 1 training
ax_tl = fig.add_subplot(gs[0,0])
ax_tl.plot(e1, stage1_train, 'b-o', ms=3, lw=1.8, label='Train')
ax_tl.plot(e1, stage1_val,   'r-s', ms=3, lw=1.8, label='Val')
ax_tl.axvline(18, color='#2e7d32', ls='--', lw=1.5,
              label='Best Ep.18')
ax_tl.set_title('Stage 1 Training Curve', fontweight='bold', fontsize=10)
ax_tl.set_xlabel('Epoch', fontsize=9)
ax_tl.set_ylabel('Loss', fontsize=9)
ax_tl.legend(fontsize=8); ax_tl.grid(True, alpha=0.3)
ax_tl.set_ylim(0,5)
ax_tl.spines['top'].set_visible(False)
ax_tl.spines['right'].set_visible(False)

# Top-middle: Ablation bars (key metrics only)
ax_tm = fig.add_subplot(gs[0,1])
sm = ['BLEU-1','ROUGE-1','ROUGE-L','Abn. F1','Recall']
bv2 = [0.0003, 0.0147, 0.0146, 0.0000, 0.0000]
av2 = [0.2884, 0.4352, 0.3103, 0.8802, 0.9795]
xp  = np.arange(len(sm))
ax_tm.bar(xp-0.2, bv2, 0.38, label='FlatMLP',
          color='#ef9a9a', edgecolor='#c62828', lw=0.8)
ax_tm.bar(xp+0.2, av2, 0.38, label='AnatomyGuided',
          color='#42a5f5', edgecolor='#0d47a1', lw=0.8)
ax_tm.set_xticks(xp)
ax_tm.set_xticklabels(sm, fontsize=8)
ax_tm.set_title('Ablation: Key Metrics', fontweight='bold', fontsize=10)
ax_tm.set_ylabel('Score', fontsize=9)
ax_tm.legend(fontsize=8); ax_tm.grid(True, axis='y', alpha=0.3)
ax_tm.set_ylim(0,1.2)
ax_tm.spines['top'].set_visible(False)
ax_tm.spines['right'].set_visible(False)

# Top-right: Mini radar
ax_tr = fig.add_subplot(gs[0,2], polar=True)
ax_tr.set_facecolor('#fafafa')
cats3 = ['BLEU-1','ROUGE-1','ROUGE-L','Abn.F1','Recall']
N3    = len(cats3)
ang3  = [n/float(N3)*2*np.pi for n in range(N3)]+[0]
br3   = [0.0003,0.0147,0.0146,0.0000,0.0000,0.0003]
ar3   = [0.2884,0.4352,0.3103,0.8802,0.9795,0.2884]
ax_tr.plot(ang3, br3, 'o-', lw=2, color='#e53935', label='FlatMLP')
ax_tr.fill(ang3, br3, alpha=0.12, color='#e53935')
ax_tr.plot(ang3, ar3, 'o-', lw=2, color='#1565c0', label='Ours')
ax_tr.fill(ang3, ar3, alpha=0.15, color='#1565c0')
ax_tr.set_xticks(ang3[:-1])
ax_tr.set_xticklabels(cats3, size=8, fontweight='bold')
ax_tr.set_ylim(0,1)
ax_tr.set_title('Radar', fontweight='bold', fontsize=10, pad=15)
ax_tr.legend(loc='upper right', bbox_to_anchor=(1.5,1.1), fontsize=8)
ax_tr.grid(color='grey', ls='--', lw=0.4, alpha=0.5)

# Bottom: Clinical F1
ax_b = fig.add_subplot(gs[1,:])
col2 = ['#ef5350' if f<0.3 else '#ffa726' if f<0.5 else '#66bb6a'
        for f in clinical_f1]
ax_b.bar(clinical_labels, clinical_f1, color=col2,
         edgecolor='#333', linewidth=0.8)
for i, (lbl, val) in enumerate(zip(clinical_labels, clinical_f1)):
    ax_b.text(i, val+0.012, f'{val:.3f}', ha='center', va='bottom',
              fontsize=7.5, fontweight='bold')
ax_b.axhline(0.3321, color='#1565c0', ls='--', lw=1.8,
             label='Macro-F1=0.3321')
ax_b.axhline(0.5897, color='#880e4f', ls='--', lw=1.8,
             label='Micro-F1=0.5897')
ax_b.set_title('Clinical Entity F1 per Finding',
               fontweight='bold', fontsize=10)
ax_b.set_ylabel('F1 Score', fontsize=9)
ax_b.legend(fontsize=9, loc='upper right')
ax_b.set_ylim(0,1.0)
ax_b.grid(True, axis='y', alpha=0.3)
ax_b.tick_params(axis='x', labelsize=7.5)
ax_b.spines['top'].set_visible(False)
ax_b.spines['right'].set_visible(False)

fig.suptitle('BanglaCXR — Complete Evaluation Dashboard',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig(os.path.join(SAVE_DIR,'fig6_dashboard.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f'✓ {os.path.join(SAVE_DIR, "fig6_dashboard.png")}')

print(f'\n{"="*50}')
print(f'All 6 figures saved to: {SAVE_DIR}')
print(f'{"="*50}')
print('fig1_training_curves.png  → Section 4.1')
print('fig2_ablation_bars.png    → Section 4.2 (main novelty proof)')
print('fig3_architecture.png     → Section 3 Methods (Figure 1)')
print('fig4_radar.png            → Section 4.2')
print('fig5_clinical_f1.png      → Section 4.3')
print('fig6_dashboard.png        → Appendix / Poster')

Bengali font found: Nirmala UI
✓ C:\Users\faisal\Downloads\Ahasan\Thesis\BanglaCXR_Ablation\figures\fig1_training_curves.png


C:\Users\faisal\AppData\Local\Temp\ipykernel_13360\1935522143.py:145: UserWarning: Glyph 8776 (\N{ALMOST EQUAL TO}) missing from font(s) Nirmala UI.
  plt.tight_layout()
C:\Users\faisal\AppData\Local\Temp\ipykernel_13360\1935522143.py:147: UserWarning: Glyph 8776 (\N{ALMOST EQUAL TO}) missing from font(s) Nirmala UI.
  plt.savefig(p2, dpi=300, bbox_inches='tight', facecolor='white')


✓ C:\Users\faisal\Downloads\Ahasan\Thesis\BanglaCXR_Ablation\figures\fig2_ablation_bars.png


C:\Users\faisal\AppData\Local\Temp\ipykernel_13360\1935522143.py:220: UserWarning: Glyph 9733 (\N{BLACK STAR}) missing from font(s) Nirmala UI.
  plt.tight_layout()
C:\Users\faisal\AppData\Local\Temp\ipykernel_13360\1935522143.py:222: UserWarning: Glyph 9733 (\N{BLACK STAR}) missing from font(s) Nirmala UI.
  plt.savefig(p3, dpi=300, bbox_inches='tight', facecolor='white')


✓ C:\Users\faisal\Downloads\Ahasan\Thesis\BanglaCXR_Ablation\figures\fig3_architecture.png
✓ C:\Users\faisal\Downloads\Ahasan\Thesis\BanglaCXR_Ablation\figures\fig4_radar.png


C:\Users\faisal\AppData\Local\Temp\ipykernel_13360\1935522143.py:307: UserWarning: Glyph 8804 (\N{LESS-THAN OR EQUAL TO}) missing from font(s) Nirmala UI.
  plt.tight_layout()
C:\Users\faisal\AppData\Local\Temp\ipykernel_13360\1935522143.py:307: UserWarning: Glyph 8805 (\N{GREATER-THAN OR EQUAL TO}) missing from font(s) Nirmala UI.
  plt.tight_layout()
C:\Users\faisal\AppData\Local\Temp\ipykernel_13360\1935522143.py:309: UserWarning: Glyph 8804 (\N{LESS-THAN OR EQUAL TO}) missing from font(s) Nirmala UI.
  plt.savefig(p5, dpi=300, bbox_inches='tight', facecolor='white')
C:\Users\faisal\AppData\Local\Temp\ipykernel_13360\1935522143.py:309: UserWarning: Glyph 8805 (\N{GREATER-THAN OR EQUAL TO}) missing from font(s) Nirmala UI.
  plt.savefig(p5, dpi=300, bbox_inches='tight', facecolor='white')


✓ C:\Users\faisal\Downloads\Ahasan\Thesis\BanglaCXR_Ablation\figures\fig5_clinical_f1.png
✓ C:\Users\faisal\Downloads\Ahasan\Thesis\BanglaCXR_Ablation\figures\fig6_dashboard.png

All 6 figures saved to: C:\Users\faisal\Downloads\Ahasan\Thesis\BanglaCXR_Ablation\figures
fig1_training_curves.png  → Section 4.1
fig2_ablation_bars.png    → Section 4.2 (main novelty proof)
fig3_architecture.png     → Section 3 Methods (Figure 1)
fig4_radar.png            → Section 4.2
fig5_clinical_f1.png      → Section 4.3
fig6_dashboard.png        → Appendix / Poster
